# Delta Lake Basics Lab

In this notebook, we will practice three core Delta Lake concepts:

1. **Table Versioning** — Create a table, make changes, and review history.
2. **Incremental Loading** — Use `COPY INTO` to load files in batches without reprocessing.
3. **Time Travel** — Query old versions of data using `VERSION AS OF` and `TIMESTAMP AS OF`.

---

## What is Delta Lake?

Delta Lake is a storage layer that sits on top of your data lake. It brings **ACID transactions**, **versioning**, and **time travel** to big data — things normally found only in databases.

| Feature | What It Means |
|---------|---------------|
| ACID | Your data changes are safe and reliable. |
| Versioning | Every change creates a new version you can see. |
| Time Travel | You can look at data as it was in the past. |
| COPY INTO | Load data from files without duplicating rows. |

In [0]:
%sql
-- Create a database to keep our work organized
CREATE DATABASE IF NOT EXISTS cyntexa_dev.delta_lab;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS cyntexa_dev.delta_lab.employees 
AS SELECT * EXCEPT (_rescued_data)
FROM
read_files(
    "/Volumes/cyntexa_dev/delta_lab/raw/Employees/",
    format => 'csv',
    header => true,
    inferSchema => true
    );

In [0]:
%sql
SELECT * FROM cyntexa_dev.delta_lab.employees;

In [0]:
%sql
INSERT INTO cyntexa_dev.delta_lab.employees VALUES
(4, 'Diana', 'HR', 70000);

SELECT * FROM cyntexa_dev.delta_lab.employees;


In [0]:
%sql
UPDATE cyntexa_dev.delta_lab.employees
SET SALARY = 95000
WHERE name = 'Alice';

-- Check the result
SELECT * FROM cyntexa_dev.delta_lab.employees;

In [0]:
%sql
-- Add another new employee
INSERT INTO cyntexa_dev.delta_lab.employees
VALUES (5, 'Eve', 'Engineering', 88000);

-- Check the result
SELECT * FROM cyntexa_dev.delta_lab.employees;

In [0]:
%sql
DESCRIBE HISTORY cyntexa_dev.delta_lab.employees;

## Task 2: Incremental Loading with COPY INTO

In real projects, data arrives in files over time (e.g., every hour or daily).  
We need to load new files without reloading old ones.

**COPY INTO** is perfect for this:
- It remembers which files it already loaded.
- If you run it again on the same file, it skips it.
- Only new files get inserted.

We will:
1. Create an empty bronze table.
2. Run COPY INTO for Batch 1.
3. Run COPY INTO again for Batch 1 + Batch 2.
4. Prove Batch 1 was NOT loaded twice.

In [0]:
%sql
CREATE TABLE cyntexa_dev.delta_lab.bronze_orders;

In [0]:
%sql
-- Load the first batch of files
COPY INTO cyntexa_dev.delta_lab.bronze_orders
FROM '/Volumes/cyntexa_dev/delta_lab/raw/Batch 1 /'
FILEFORMAT = CSV 
FORMAT_OPTIONS ('header' = 'true' , 'inferSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true')

In [0]:
%sql
SELECT * FROM cyntexa_dev.delta_lab.bronze_orders;

In [0]:
%sql
COPY INTO cyntexa_dev.delta_lab.bronze_orders
FROM '/Volumes/cyntexa_dev/delta_lab/raw/Batch 1 /'
FILEFORMAT = CSV 
FORMAT_OPTIONS ('header' = 'true' , 'inferSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true')

In [0]:
%sql
SELECT COUNT(*) AS total_rows FROM cyntexa_dev.delta_lab.bronze_orders;

In [0]:
%sql
-- Now load the second batch
COPY INTO cyntexa_dev.delta_lab.bronze_orders
FROM
'/Volumes/cyntexa_dev/delta_lab/raw/Batch 2/'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true' , 'inferSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true')

In [0]:
%sql
-- Should now show all 4 rows
SELECT * FROM cyntexa_dev.delta_lab.bronze_orders;

In [0]:
%sql
-- Should show 4 total
SELECT COUNT(*) AS total_rows FROM cyntexa_dev.delta_lab.bronze_orders;

## Task 3: Time Travel

Delta Lake lets you query data as it was at any point in time.

Two ways to do this:
1. `VERSION AS OF` — pick a specific version number.
2. `TIMESTAMP AS OF` — pick a specific moment in time.

We will use the `employees` table from Task 1, which has 4 versions (0 to 3).

In [0]:
%sql
-- Version 0: When the table was first created (3 rows)
SELECT * FROM cyntexa_dev.delta_lab.employees VERSION AS OF 0;

In [0]:
%sql
-- Version 1: After first INSERT (4 rows)
SELECT * FROM cyntexa_dev.delta_lab.employees VERSION AS OF 1;

In [0]:
%sql
-- Version 2: After UPDATE (Alice got a raise)
SELECT * FROM cyntexa_dev.delta_lab.employees VERSION AS OF 2;

In [0]:
%sql
-- Version 3: After second INSERT (5 rows)
SELECT * FROM cyntexa_dev.delta_lab.employees VERSION AS OF 3;

In [0]:
%sql
DESCRIBE HISTORY cyntexa_dev.delta_lab.employees;

In [0]:
%sql
SELECT * FROM cyntexa_dev.delta_lab.employees TIMESTAMP AS OF '2026-09-07T12:12:23.000+00:00'

## Summary

| Task | What We Learned |
|------|-----------------|
| **Task 1** | Delta tables track every change. `DESCRIBE HISTORY` shows the full log. |
| **Task 2** | `COPY INTO` loads files once and remembers them. Safe to rerun. |
| **Task 3** | `VERSION AS OF` and `TIMESTAMP AS OF` let you look at the past. |

## Key Concepts

- **Delta Lake** = A storage layer with database-like safety for your data lake.
- **Versioning** = Every write creates a new version. Nothing is truly lost.
- **Idempotency** = Running the same command twice gives the same result.
- **Time Travel** = Query historical data without backups or snapshots.
- **Bronze Layer** = The landing zone for raw data in a medallion architecture.

## Next Steps

- Try `RESTORE TABLE employees TO VERSION AS OF 1` to roll back changes.
- Explore `MERGE INTO` for upserts (update + insert together).
- Learn about `OPTIMIZE` and `VACUUM` for managing old file versions.

# Delta Lake Intermediate Lab

## What We Will Cover

| Task | Topic | Goal |
|------|-------|------|
| **Task 4** | Schema Evolution | Add a new column with `mergeSchema`, then change a column type with `overwriteSchema` |
| **Task 5** | Autoloader Streaming | Set up a stream that watches a folder and auto-ingests new files |
| **Task 6** | RESTORE & Time Travel | Roll back a bad schema change and understand what happens to later versions |

---

## What is Schema Evolution?

In the real world, data changes:
- Your app adds a new field (e.g., `email`).
- You realize a column was stored as the wrong type (e.g., `salary` as integer instead of decimal).

Delta Lake lets you evolve the schema safely instead of rebuilding the whole table.

---

## What is Autoloader?

Autoloader is like a smart mailman:
- It watches a folder (mailbox).
- When a new file (letter) arrives, it reads it automatically.
- It remembers what it already read — no duplicates.

---

## What is RESTORE?

RESTORE is an "Undo" button for your table. If someone makes a bad change, you can roll back to a good version.

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS cyntexa_dev.delta_intermediate;

In [0]:
%sql
-- Create table from base CSV (4 columns: id, name, department, salary)
CREATE TABLE IF NOT EXISTS cyntexa_dev.delta_intermediate.emp_schema_demo
AS SELECT * EXCEPT(_rescued_data)
FROM read_files(
    "/Volumes/cyntexa_dev/delta_lab/raw/Employees Intermediate/",
    format => 'csv',
    header => 'true',
    inferSchema => 'true'
);

-- Check the schema
DESCRIBE cyntexa_dev.delta_intermediate.emp_schema_demo;

In [0]:
%sql
-- Check the data
SELECT * FROM cyntexa_dev.delta_intermediate.emp_schema_demo;

## Part A: mergeSchema — Adding a New Column

**Scenario:** Your data source now includes an `email` column. You want to add it to the table WITHOUT breaking anything.

**mergeSchema** does this:
- Keeps all existing columns (id, name, department, salary).
- Adds the new column (`email`) from the incoming data.
- Fills missing values with `NULL` for old rows.

This is **safe and non-destructive**.

In [0]:
%sql
-- Append new data that has an extra 'email' column
-- mergeSchema = true tells Delta: "If you see new columns, add them"
INSERT INTO cyntexa_dev.delta_intermediate.emp_schema_demo
SELECT * FROM csv.`/Volumes/cyntexa_dev/delta_lab/raw/Employee With Email/`;

-- Oops! This will FAIL because the CSV has 5 columns but our table has 4.
-- We need to use PySpark with mergeSchema option instead.
-- A metadata mismatch was detected when writing to the Delta table.

In [0]:
# Read the new data that has the extra 'email' column
df_new = spark.read.csv("/Volumes/cyntexa_dev/delta_lab/raw/Employee With Email/",
                        header = True,
                        inferSchema = True)

# Show what the new data looks like
df_new.show()

In [0]:
# Write to the existing table WITH mergeSchema
# This adds the 'email' column to the table automatically

# Write to the existing table WITH mergeSchema
# This adds the 'email' column to the table automatically
df_new.write\
      .mode("append")\
      .option("mergeSchema" , "true")\
      .saveAsTable("cyntexa_dev.delta_intermediate.emp_schema_demo")

In [0]:
%sql
-- Check the schema now has 5 columns
DESCRIBE cyntexa_dev.delta_intermediate.emp_schema_demo;


## Part B: overwriteSchema — Changing an Existing Column's Type

**Scenario:** You realize `salary` should store decimals (e.g., 95000.50), not just whole numbers. The current type is BIGINT. You need to change it to DOUBLE.

**overwriteSchema** does this:
- Replaces the ENTIRE table schema with the new data's schema.
- Can change column types, remove columns, or reorder them.
- **Warning:** This is powerful and can break things. Use carefully.

**When to use:**
- Fixing a wrong column type.
- Rebuilding a table with a cleaner schema.

**When NOT to use:**
- When you just want to add a column (use mergeSchema instead).
- In production without testing first.

In [0]:
# Read data where salary is a decimal (DOUBLE type)
df_decimal = spark.read.csv("/Volumes/cyntexa_dev/delta_lab/raw/Employee Double Salary/",
                            header = True,
                            inferSchema = True)
# Check the schema — notice salary is now DOUBLE
df_decimal.printSchema()
# Write with overwriteSchema to change the table's schema
df_decimal.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("cyntexa_dev.delta_intermediate.emp_schema_demo")

In [0]:
from pyspark.sql.functions import col 

# Read the original CSV (salary is still INT/BIGINT)
df_old = spark.read.csv("/Volumes/cyntexa_dev/delta_lab/raw/Employees Intermediate/",
                        header = True,
                        inferSchema = True)

# Cast salary to DOUBLE so it matches the new table schema
df_old = df_old.withColumn("salary" , col("salary").cast("double"))

df_old.write\
      .mode("append")\
      .saveAsTable("cyntexa_dev.delta_intermediate.emp_schema_demo")

In [0]:
%sql
-- Check the schema: salary should now be DOUBLE
DESCRIBE cyntexa_dev.delta_intermediate.emp_schema_demo;

In [0]:
%sql
-- Check the data
SELECT * FROM cyntexa_dev.delta_intermediate.emp_schema_demo;


## Summary: mergeSchema vs overwriteSchema

| | mergeSchema | overwriteSchema |
|---|---|---|
| **What it does** | Adds new columns from incoming data | Replaces the entire schema |
| **Existing columns** | Kept as-is | Can be changed, removed, or reordered |
| **Column types** | Cannot change existing types | Can change existing types |
| **Safety** | Safe — non-destructive | Risky — can break downstream code |
| **Use case** | New field added to source data | Fixing a wrong column type |
| **Example** | Adding `email` column | Changing `salary` from INT to DOUBLE |

### Analogy

Think of a table as a form:

- **mergeSchema** = Someone adds a new field to the form. You add a blank box for it. Old forms stay the same.
- **overwriteSchema** = You redesign the entire form. You can change box sizes, remove fields, or move them around.

In [0]:
from pyspark.sql.functions import * 

# Set up the Autoloader stream
# cloudFiles tells Spark to use Databricks Autoloader
stream_df = (
    spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format" , "csv")
            .option("cloudFiles.schemaLocation" , "/Volumes/cyntexa_dev/delta_lab/raw/schema_checkpoints")
            .option("inferSchema" , "true")
            .load("/Volumes/cyntexa_dev/delta_lab/raw/landing_zone/")
)

In [0]:
# Write the stream to our Delta table
# checkpointLocation saves stream progress so we don't reprocess files
query = (
    stream_df.writeStream
                 .outputMode("append")
                 .option("checkpointLocation" , "/Volumes/cyntexa_dev/delta_lab/raw/stream_checkpoints")
                 .trigger(availableNow=True)
                 .toTable("cyntexa_dev.delta_intermediate.events_stream")
)

print("Stream started! Query name:", query.name)

In [0]:
query.stop()

In [0]:
%sql
SELECT * FROM cyntexa_dev.delta_intermediate.events_stream

## Autoloader Summary

### What Happened
1. We started a stream watching `/landing_zone`.
2. We dropped `batch1.csv` — stream loaded it.
3. We dropped `batch2.csv` — stream detected it automatically and loaded it.
4. We dropped `batch3.csv` — same thing.
5. No duplicates. No manual intervention.

### Why This Matters
In production, data arrives continuously:
- Log files every minute
- Sensor data every second
- Partner uploads every hour

Autoloader handles this without you writing cron jobs or file-watching scripts.

### Key Components
| Component | Purpose |
|-----------|---------|
| **cloudFiles** | The Autoloader format |
| **schemaLocation** | Saves inferred schema across restarts |
| **checkpointLocation** | Tracks which files were already processed |
| **writeStream** | Starts the continuous processing |

In [0]:
%sql
-- Create a fresh table for this task
CREATE TABLE IF NOT EXISTS cyntexa_dev.delta_intermediate.emp_restore_demo
AS SELECT * EXCEPT(_rescued_data)
FROM read_files(
    "/Volumes/cyntexa_dev/delta_lab/raw/Employees Intermediate/",
    format => "csv",
    header => "true",
    inferSchema => "true"
);

-- Version 0: Initial table (3 rows)
SELECT * FROM cyntexa_dev.delta_intermediate.emp_restore_demo;

In [0]:
%sql
-- Version 1: Add a new employee
INSERT INTO cyntexa_dev.delta_intermediate.emp_restore_demo VALUES
(4, 'Diana', 'HR', 70000);

-- Version 2: Give Alice a raise
UPDATE cyntexa_dev.delta_intermediate.emp_restore_demo 
SET salary = 95000 WHERE name = 'Alice';

-- Check history so far
DESCRIBE HISTORY cyntexa_dev.delta_intermediate.emp_restore_demo;

In [0]:
# Simulate a BAD change: someone overwrites the schema incorrectly
# This data has different column names — a mistake!

df_bad = spark.createDataFrame(
    [(999, "Hacker", "IT", 1000000)],
    ["emp_id", "emp_name", "dept", "pay"]  # Wrong column names!
)

# This would be a disaster in production
df_bad.write\
      .mode("append")\
      .option("mergeSchema" , "true")\
      .saveAsTable("cyntexa_dev.delta_intermediate.emp_restore_demo")

In [0]:
%sql
SELECT * FROM cyntexa_dev.delta_intermediate.emp_restore_demo

In [0]:
%sql
-- Check the full history
DESCRIBE HISTORY cyntexa_dev.delta_intermediate.emp_restore_demo;

In [0]:
%sql
-- Restore to Version 2 (before the bad schema change)
-- Replace '2' with the actual version number from your DESCRIBE HISTORY
RESTORE TABLE cyntexa_dev.delta_intermediate.emp_restore_demo TO VERSION AS OF 2;

-- Check the result
SELECT * FROM cyntexa_dev.delta_intermediate.emp_restore_demo;

In [0]:
%sql
-- Look at the history AFTER restore
DESCRIBE HISTORY cyntexa_dev.delta_intermediate.emp_restore_demo;

## What Happens to Versions After the Restore Point?

### The Short Answer
**Nothing happens to them.** They still exist.

### The Long Answer

| Question | Answer |
|----------|--------|
| Are Versions 3+ deleted? | **No.** They remain in the history log. |
| Can I still query them? | **Yes.** `VERSION AS OF 3` still works. |
| What changed? | The **current** table data was rolled back. |
| What's the new version? | A new version (e.g., Version 4) is created recording the RESTORE. |

## Lab Summary

### Task 4: Schema Evolution
- `mergeSchema` = Add new columns safely (non-destructive).
- `overwriteSchema` = Change the whole schema (powerful, use with care).
- **mergeSchema** is for growth. **overwriteSchema** is for fixes.

### Task 5: Autoloader Streaming
- Autoloader (`cloudFiles`) watches a folder and auto-processes new files.
- Checkpoints remember what was already read — no duplicates.
- Perfect for real-time data ingestion without manual jobs.

### Task 6: RESTORE
- `RESTORE TABLE ... TO VERSION AS OF` rolls back table data.
- Old versions are NOT deleted from history.
- A new version is created to record the restore action.
- Time travel still works for all historical versions.

---

## Key Takeaways

| Concept | One-Line Summary |
|---------|-----------------|
| Delta Lake | A storage layer that makes your data lake as reliable as a database. |
| Schema Evolution | Change table structure without rebuilding everything. |
| mergeSchema | "Add what's new, keep what's old." |
| overwriteSchema | "Replace the blueprint entirely." |
| Autoloader | "Set it and forget it" file ingestion. |
| RESTORE | "Undo" for your entire table. |
| Time Travel | The past is never truly gone. |

---

# 3. Advance Tasks 
# Databricks Ingestion Patterns Comparison & Recommendation

---

## 1. Introduction
When loading data into Databricks, choosing the right ingestion pattern is critical for **cost**, **latency**, and **operational complexity**. 

Below is a simple comparison of the four main ingestion patterns:
1. **Batch CTAS** (`CREATE TABLE AS SELECT`)
2. **COPY INTO**
3. **Auto Loader** (`cloudFiles`)
4. **Lakeflow Declarative Pipelines** (Delta Live Tables / Lakeflow)

---

## 2. Comparison Matrix

| Ingestion Pattern | Cost | Latency | Operational Complexity | Best Use Case |
| :--- | :--- | :--- | :--- | :--- |
| **Batch CTAS** | High (for frequent runs) | High (Hours / Daily) | High (Manual tracking required) | One-time loads or full history refresh |
| **COPY INTO** | Low to Medium | Medium (Scheduled / Near Real-Time) | Low (Built-in file tracking) | Simple, scheduled file loading |
| **Auto Loader** | Low | Low (Seconds to Minutes) | Medium (Requires streaming setup) | Unpredictable or high-volume streaming files |
| **Lakeflow Pipelines** | Medium | Very Low (Real-time / Near Real-Time) | Low (Declarative UI & auto-managed) | Enterprise end-to-end data pipelines |

---

## 3. Deep Dive into the 4 Patterns

### A. Batch CTAS (`CREATE TABLE AS SELECT`)
* **How it works:** Reads source files and overwrites or appends to a target table using standard SQL.
* **Cost:** **High** if run frequently because it rescans data or spins up full clusters for simple batch jobs.
* **Latency:** **High** (Runs on fixed schedules like once a day or once an hour).
* **Operational Complexity:** **High**. You must manually write logic to avoid loading duplicate files (file tracking).
* **Summary:** Good for simple, one-time bulk loads, but bad for frequent file processing.

---

### B. COPY INTO
* **How it works:** An idempotent SQL command that incrementally loads new files from cloud storage into Delta tables.
* **Cost:** **Low to Medium**. You only pay for compute when the command runs.
* **Latency:** **Medium**. Best used with scheduled jobs (e.g., every 15 minutes or every hour).
* **Operational Complexity:** **Low**. Databricks automatically tracks which files were already loaded, preventing duplicates.
* **Summary:** Great for simple scheduled jobs when files arrive at fixed intervals.

---

### C. Auto Loader (`cloudFiles`)
* **How it works:** Uses Databricks Structured Streaming to discover and process new files automatically as they land in cloud storage.
* **Cost:** **Low**. Efficient file discovery (uses cloud notification services or directory listing) without scanning entire folders.
* **Latency:** **Low** (Processes files in seconds or minutes).
* **Operational Complexity:** **Medium**. Requires writing streaming pipelines (`readStream` / `writeStream`) and managing checkpoints.
* **Summary:** The industry standard for continuous, unpredictable file arrival.

---

### D. Lakeflow Declarative Pipelines (Delta Live Tables)
* **How it works:** A fully managed, declarative framework where you define target tables using simple SQL or Python, and Databricks manages the underlying infrastructure, DAGs, and retries.
* **Cost:** **Medium**. Uses specialized DLT compute, but saves significant engine orchestration costs and engineering hours.
* **Latency:** **Very Low**. Supports continuous streaming or triggered execution.
* **Operational Complexity:** **Low**. Automatic data quality checks (expectations), tracking, monitoring, and error handling out-of-the-box.
* **Summary:** Best for full end-to-end data pipelines that require data quality, governance, and low maintenance.

---

## 4. Recommendation for Cyntexa

### Scenario:
* **Source:** Files arrive **unpredictably** throughout the day.
* **Goal:** Lowest cost, low latency, and low operational overhead.

### Recommended Pattern: **Databricks Auto Loader** *(or Lakeflow Pipelines with Auto Loader underlying engine)*

#### Why Auto Loader is the Best Choice for Cyntexa:

1. **Handles Unpredictable File Arrival Efficiently:**
   * Batch CTAS or COPY INTO require periodic polling (e.g., running every 5 minutes), which wastes cluster compute time when no new files are present.
   * Auto Loader can run in **Trigger Once / AvailableNow** mode or continuous mode, so it only processes data when files actually arrive.

2. **Cost-Effective File Discovery:**
   * As folder sizes grow to millions of files, standard directory listing becomes slow and expensive.
   * Auto Loader uses **Cloud Notification mode** (AWS SQS, Azure Event Grid, or GCP Pub/Sub) to automatically detect new files without scanning the folder structure.

3. **Schema Evolution & Safety:**
   * If source files suddenly change structure (e.g., a new column is added unexpectedly), Auto Loader handles **schema drift** automatically without breaking the pipeline.

4. **Low Operational Complexity:**
   * It provides built-in checkpointing and **exactly-once processing guarantees**, meaning files are never processed twice even if the cluster restarts.

---

## 5. Summary Cheat Sheet for Cyntexa

* **Use CTAS when:** Loading standard lookup tables once.
* **Use COPY INTO when:** Running a simple scheduled batch job on a small dataset.
* **Use Auto Loader when:** **Files arrive unpredictably and continuously throughout the day (Recommended).**
* **Use Lakeflow Pipelines when:** You need full end-to-end lineage, data quality rules, and automated pipeline monitoring across multiple tables.


# Task 9 
# 2 AM Recovery Runbook: Corrupted Silver Table

---

## Recovery Path A: RESTORE (Fastest — Recommended for Emergencies)

Use this when you want to **roll the entire table back** to a known good state. This is the "big red undo button."

### Step 1: Inspect the Damage

```sql
-- See what the table looks like RIGHT NOW (the corrupted state)
SELECT * FROM cyntexa_dev.silver_orders LIMIT 20;

-- Count total rows to see if the bad file inflated the table
SELECT COUNT(*) AS current_row_count FROM cyntexa_dev.silver_orders;
```

**What this tells you:**  
- Are there nulls where there shouldn't be?  
- Are there impossible values (e.g., negative prices, dates in the future)?  
- Did the row count jump unexpectedly?

---

### Step 2: Check the History

```sql
-- See every change ever made to this table, with timestamps
DESCRIBE HISTORY cyntexa_dev.silver_orders;
```

**What to look for in the output:**

| Column | What It Means | What to Check |
|--------|---------------|---------------|
| `version` | The save file number | Find the last version BEFORE 2 AM |
| `timestamp` | When the change happened | Look for the batch that ran around 2:00 AM |
| `operation` | What type of change | `WRITE`, `MERGE`, `UPDATE`, etc. |
| `operationParameters` | Details of the change | Look for the bad file path or batch ID |
| `numOutputRows` | Rows after the change | A sudden jump means bad data was added |
| `isolationLevel` | Transaction safety | Should be `Serializable` |

**Example of what you might see:**

| version | timestamp | operation | numOutputRows |
|---------|-----------|-----------|---------------|
| 0 | 2026-09-07 18:00 | CREATE TABLE AS SELECT | 50,000 |
| 1 | 2026-09-07 20:00 | WRITE | 52,000 |
| 2 | 2026-09-07 22:00 | WRITE | 55,000 |
| 3 | 2026-09-08 02:00 | WRITE | 250,000 | ← **SUSPICIOUS! Row count exploded.** |
| 4 | 2026-09-08 02:15 | WRITE | 250,000 | ← More bad data |

**Your target:** Version `2` (the last good state before the 2 AM corruption).

---

### Step 3: Verify the Good Version

**Never restore blindly. Always peek first.**

```sql
-- Look at the table as it was at Version 2 (the good state)
SELECT * FROM cyntexa_dev.silver_orders VERSION AS OF 2 LIMIT 20;

-- Check row count at that version
SELECT COUNT(*) AS good_row_count FROM cyntexa_dev.silver_orders VERSION AS OF 2;

-- Check a key metric (e.g., total revenue) to confirm it's sane
SELECT SUM(order_amount) AS total_revenue FROM cyntexa_dev.silver_orders VERSION AS OF 2;
```

**What this tells you:**  
- Does this version look clean?  
- Is the row count reasonable?  
- Are the business metrics making sense?

---

### Step 4: Execute the RESTORE

```sql
-- Roll the table back to Version 2
RESTORE TABLE cyntexa_dev.silver_orders TO VERSION AS OF 2;
```

**What happens under the hood:**
- The table's **current data** is rewound to match Version 2.
- The table's **schema** is also rewound to match Version 2.
- A **new version** is created (e.g., Version 5) that records: *"RESTORE happened here."*
- Versions 3 and 4 are **NOT deleted** — they remain in history.

---

### Step 5: Confirm the Fix

```sql
-- The table should now look exactly like Version 2
SELECT * FROM cyntexa_dev.silver_orders LIMIT 20;

-- Row count should be back to normal
SELECT COUNT(*) FROM cyntexa_dev.silver_orders;

-- Check the new history entry
DESCRIBE HISTORY cyntexa_dev.silver_orders;
```

**What to verify:**
- Row count matches the good version.
- Data looks clean.
- A new `RESTORE` entry appears in history.

---

### Step 6: Re-run the Pipeline (Carefully)

```sql
-- If the silver table is fed by a bronze table, check if bronze is also corrupted
DESCRIBE HISTORY cyntexa_dev.bronze_orders;

-- If bronze is clean, re-run the silver transformation job
-- (This depends on your pipeline tool — could be a notebook job, a DLT pipeline, or a scheduled workflow)
```

**Important:** Find and quarantine the bad file before re-running. Otherwise, the corruption will happen again.

---

# 9. Data Freshness Report Using DESCRIBE HISTORY

## What Is This About?

Your business stakeholder says: *"I was promised this table updates every hour. Is that actually happening?"*

This report uses `DESCRIBE HISTORY` to find the **truth**. It shows exactly when a table was updated, how long the gaps were between updates, and whether the SLA promise is being kept or broken.

---

## The Problem in Simple Words

| What the Stakeholder Was Told | What Might Actually Be Happening |
|------------------------------|----------------------------------|
| "The sales table refreshes every 60 minutes." | The pipeline fails at night and the table doesn't update for 6 hours. |
| "Orders data is real-time." | Files arrive every 15 minutes, but the job only runs every 2 hours. |
| "Dashboard data is always fresh." | Weekend updates are skipped to save costs. |

**DESCRIBE HISTORY** is like a **attendance log** for your table. It records every single time someone (or some job) touched the table. We can read this log and calculate: *"How long did the table sit without any updates?"*

---

## What DESCRIBE HISTORY Gives You

When you run:

```sql
DESCRIBE HISTORY your_table;
```

You get a log that looks like this:

| version | timestamp | operation | operationParameters | numOutputRows |
|---------|-----------|-----------|---------------------|---------------|
| 0 | Sep 7, 10:00 AM | CREATE TABLE AS SELECT | {query: SELECT * FROM raw} | 10,000 |
| 1 | Sep 7, 11:00 AM | WRITE | {mode: Append} | 10,500 |
| 2 | Sep 7, 12:00 PM | WRITE | {mode: Append} | 10,900 |
| 3 | Sep 7, 02:00 PM | WRITE | {mode: Append} | 11,200 |
| 4 | Sep 7, 06:00 PM | WRITE | {mode: Append} | 11,500 |

**Look at the gaps:**
- 10:00 AM → 11:00 AM = 1 hour ✅ (Good)
- 11:00 AM → 12:00 PM = 1 hour ✅ (Good)
- 12:00 PM → 02:00 PM = **2 hours** ❌ (Missed the 1 PM update!)
- 02:00 PM → 06:00 PM = **4 hours** ❌ (Big gap!)

**This is your evidence.**

---

## The Freshness Metrics You Need

| Metric | What It Means | Why It Matters |
|--------|---------------|----------------|
| **Time Since Last Update** | How many minutes/hours ago was the last write? | Stakeholders want to know: *"Is my dashboard showing old data right now?"* |
| **Average Gap Between Updates** | Average time between two consecutive writes. | Tells you the typical freshness of the table. |
| **Maximum Gap (Worst Case)** | The longest time the table went without an update. | This is usually what breaks the SLA. |
| **Update Frequency** | How many updates happened per day/week? | Shows if the pipeline is consistent or erratic. |
| **Failed Update Windows** | How many expected updates were missed? | Direct proof of SLA violation. |

---